# 00 · Setup and global settings

This notebook holds everything that the other notebooks share: the environment check, the **global settings** (the master folder and every path below it),
the constants of the study, and the functions that more than one notebook needs.

Every other notebook starts with

```python
%run ./00_setup_and_settings.ipynb
```

so that all settings and shared functions are defined in this notebook only. To use the code on your computer, edit the cell
**Global settings** below (or set the environment variable `STUDY_MASTER`).

## Order of the notebooks

| Notebook | What it does | Needs |
|---|---|---|
| `00_setup_and_settings` | environment check, global settings, constants, shared functions | – |
| `01_data_download_and_preprocessing` | where to obtain the datasets, download, waveform arrays, machine measurements, quality control, inputs of the models | internet; about 170 GB of disk for the raw data (plus about 85 GB of ZIP files during the download) and 170 GB for the arrays; PhysioNet credentialed access for MIMIC-IV and MIMIC-IV-ED |
| `02_linkage_cohort_and_sets` | clinical context of each ECG (care setting, age, sex, death within 1 year), exclusions, patient-level splits, repeat-ECG sets | credentialed data |
| `03_models_and_training` | pretrained weights, frozen embeddings with ridge regression, fine-tuning on MIMIC-IV-ECG and on CODE-15% | one GPU |
| `04_predictions_and_accuracy` | AI-ECG age of the test-split ECGs with the ten models; accuracy in MIMIC-IV-ECG, CODE-15% and PTB-XL | one GPU |
| `05_reliability_analysis` | retest reliability, sources of within-patient variation, night–daytime comparison, predicted sex, pull toward the mean age | – |
| `06_mortality_analysis` | association of the age gap with death within 1 year, SIMEX, averaging of repeat ECGs, flags | – |
| `07_tables` | Tables 1–2 and Supplementary Tables 1–4 | – |
| `08_figures` | Figs. 1–4 and Supplementary Figs. 1–6 | – |

Data, pretrained weights and trained models are **not** part of the repository. Notebook 01 explains where each dataset comes from and what access it requires;
notebook 03 does the same for the pretrained models.

**Credentialed data.** MIMIC-IV and MIMIC-IV-ED may only be used by credentialed users under the PhysioNet data use agreement.
Everything derived from them stays inside `MASTER`; the notebooks print aggregate numbers only, and counts between 1 and 9 are shown as "<10".

## Environment

The code runs with Python 3.12 and the packages listed in `environment.yml` (in the root of the repository); notebooks 03 and 04 also need one CUDA
graphics card. The run times given in the notebooks refer to one desktop computer with Windows 11 and one NVIDIA GeForce RTX 5090 graphics card.
The cell below prints the versions that are installed, so that they can be compared with `environment.yml`.

In [ ]:
import importlib
import platform

print("python", platform.python_version())
for _name in ("numpy", "pandas", "scipy", "sklearn", "matplotlib", "pyarrow", "h5py", "wfdb", "torch", "transformers"):
    try:
        print(f"{_name:13s}", importlib.import_module(_name).__version__)
    except Exception as _e:                                    # a missing package is reported, not raised: notebooks that do not need it still run
        print(f"{_name:13s} not available ({type(_e).__name__})")

## Global settings

`MASTER` is the folder that holds the data, the models and the results. It can be anywhere, also on another drive than the repository. Nothing is written outside `MASTER`.

```
MASTER/
├── data/raw/                  downloaded datasets, never modified                          (notebook 01)
├── data/derived/              waveform arrays, measurements, model inputs                   (notebook 01)
├── data/derived/credentialed/ everything derived from MIMIC-IV / MIMIC-IV-ED                 (notebook 02)
├── data/derived/sets/         cohort, splits, repeat-ECG pairs, counts                      (notebook 02)
├── data/derived/embeddings/   frozen embeddings of the foundation models                    (notebook 03)
├── models/pretrained/         pretrained weights as released by their developers            (notebook 03)
├── models/trained/            one folder per trained model                                  (notebook 03)
├── results/predictions/       AI-ECG age of the test-split ECGs, one file per model          (notebook 04)
├── results/training/          settings and validation history of every trained model         (notebook 03)
├── results/summary/           outcome tables with confidence intervals                       (notebooks 04–06)
└── output/                    tables and figures of the paper                                (notebooks 07, 08)
```

Third-party code (the model definitions of the pretrained models) is not copied into this repository. Clone it as described in notebook 03 into `THIRD_PARTY`.

In [ ]:
# ============================== GLOBAL SETTINGS ==============================
import os
from pathlib import Path

# --- the master folder: edit this line, or set the environment variable STUDY_MASTER ----------------------------------
MASTER = Path(os.environ.get("STUDY_MASTER", "path/to/master_folder"))

# --- raw data (downloaded in notebook 01) -------------------------------------------------------------------------------
RAW = MASTER / "data" / "raw"
MIMIC_ECG_DIR = RAW / "mimic-iv-ecg"            # MIMIC-IV-ECG 1.0 (PhysioNet, open): record_list.csv, machine_measurements.csv, files/
MIMIC_IV_DIR = RAW / "mimic-iv"                 # MIMIC-IV 3.1 (PhysioNet, credentialed): hosp/, icu/
MIMIC_ED_DIR = RAW / "mimic-iv-ed"              # MIMIC-IV-ED 2.2 (PhysioNet, credentialed): ed/
CODE15_DIR = RAW / "code-15"                    # CODE-15% 1.0.0 (Zenodo, open): exams.csv, exams_part0.hdf5 ... exams_part17.hdf5
PTBXL_DIR = RAW / "ptb-xl"                      # PTB-XL 1.0.3 (PhysioNet, open): ptbxl_database.csv, records500/

# --- derived data (notebooks 01 and 02) ---------------------------------------------------------------------------------
DERIVED = MASTER / "data" / "derived"
MIMIC_RECORDS = DERIVED / "mimic_records.parquet"          # one row per MIMIC-IV-ECG record (row number = idx): times, machine measurements
MIMIC_QC = DERIVED / "mimic_qc.parquet"                    # per-record quality checks
MIMIC_REPORT_FLAGS = DERIVED / "mimic_report_flags.parquet"  # machine-report statements on artifact, leads and lead reversal
MIMIC_WAVE500 = DERIVED / "mimic_wave500_int16.npy"         # (N, 12, 5000) int16, 500 Hz, MIMIC lead order, 200 units per mV
MIMIC_HUBERT = DERIVED / "mimic_hubert_f16.npy"             # (N, 2, 6000) float16, HuBERT-ECG input
CODE15_RECORDS = DERIVED / "code15_records.parquet"         # one row per row of the CODE-15% arrays: exam, patient, age, sex, split (empty if excluded)
CODE15_WAVE500 = DERIVED / "code15_wave500_int16.npy"
CODE15_HUBERT = DERIVED / "code15_hubert_f16.npy"
PTBXL_RECORDS = DERIVED / "ptbxl_records.parquet"
PTBXL_WAVE500 = DERIVED / "ptbxl_wave500_int16.npy"
PTBXL_HUBERT = DERIVED / "ptbxl_hubert_f16.npy"

CRED = DERIVED / "credentialed"                             # derived from MIMIC-IV / MIMIC-IV-ED: never share
ECG_CONTEXT = CRED / "ecg_context.parquet"                  # per ECG: care setting, admission and ED stay, age, sex, death within 1 year

SETS = DERIVED / "sets"
COHORT = SETS / "cohort.parquet"                            # per record: exclusion reason, include, split (train / val / test)
COHORT_SUMMARY = SETS / "cohort_summary.json"               # counts of the flow diagram (Fig. 1b) and of the splits
PAIRS_48H = SETS / "pairs_within_48h.parquet"               # repeat-ECG sets of the test split (notebook 02)
PAIRS_1H = SETS / "pairs_within_1h.parquet"
PAIRS_OUT30 = SETS / "pairs_outpatient_30d.parquet"
PAIRS_NIGHT_DAY = SETS / "pairs_night_daytime.parquet"

EMBEDDINGS = DERIVED / "embeddings"                         # {fm}_f16.npy, (N, dim) float16, row = idx (notebook 03)

# --- models ---------------------------------------------------------------------------------------------------------------
PRETRAINED = MASTER / "models" / "pretrained"               # weights of the pretrained models as released (notebook 03)
TRAINED = MASTER / "models" / "trained"                     # one folder per trained model: {model key}/best.pt (notebook 03)
THIRD_PARTY = MASTER / "third_party"                        # clones of the developers' code repositories (notebook 03)

# --- results and output ---------------------------------------------------------------------------------------------------
RESULTS = MASTER / "results"
PREDICTIONS = RESULTS / "predictions"                       # {model key}.parquet; external/{model key}_{code15|ptbxl}.parquet
TRAINING = RESULTS / "training"                             # {model key}.json: settings, chosen penalty or checkpoint, validation history
SUMMARY = RESULTS / "summary"                               # outcome tables (notebooks 04-06), read by notebooks 07 and 08
TABLES = MASTER / "output" / "tables"
FIGURES = MASTER / "output" / "figures"

for _d in (RAW, DERIVED, CRED, SETS, EMBEDDINGS, PRETRAINED, TRAINED, THIRD_PARTY, PREDICTIONS, PREDICTIONS / "external", TRAINING,
           SUMMARY, TABLES, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)
print("MASTER =", MASTER)

## Constants of the study

These values define the design described in the Methods of the paper. They are collected here so that no notebook repeats them.

In [ ]:
# --- cohort and splits ----------------------------------------------------------------------------------------------------
AGE_MIN, AGE_MAX = 18, 88                 # ECGs recorded outside these ages (or of unknown age) are excluded before any split
MIMIC_SPLIT = {"train": 0.40, "val": 0.10, "test": 0.50}   # patient-level split of MIMIC-IV-ECG
MIMIC_SPLIT_SEED = 20261107
CODE15_SPLIT = {"train": 0.80, "val": 0.10, "test": 0.10}  # patient-level split of CODE-15%
CODE15_SPLIT_SEED = 20261119

# --- repeat-ECG sets (test split; pairs of ECGs of the same patient, earlier and later) ------------------------------------
WITHIN_48H = 48.0                         # hours: every pair within 48 hours (the main set)
WITHIN_7D = 168.0                         # hours: runs of five ECGs within 7 days (averaging of repeat ECGs)
WITHIN_1H = 1.0                           # hours: same visit and same care setting
OUTPATIENT_DAYS = 30                      # days: both outpatient, both recorded 08:00-17:59
OUTPATIENT_HOURS = (8, 17)                # first and last clock hour included
NIGHT_HOURS = (0, 5)                      # 00:00-05:59
DAYTIME_HOURS = (10, 15)                  # 10:00-15:59

# --- definitions used in the analyses --------------------------------------------------------------------------------------
OLDER_FLAG = 8.0                          # an age gap of 8 years or more flags accelerated aging (Lima et al.)
OR_PER = 10                               # odds ratios are reported per 10-year larger age gap
SMALL_CELL = 10                           # counts between 1 and SMALL_CELL - 1 are printed as "<10"

# --- resampling -------------------------------------------------------------------------------------------------------------
N_BOOT = 1000                             # patient bootstrap resamples for every confidence interval
BOOT_SEED = 2026                          # every analysis derives its own generator from this seed

# --- amplitude scale of the Lima et al. model --------------------------------------------------------------------------------
# The Lima et al. model was trained on CODE ECGs whose amplitude units differ from millivolts. Tracings in mV are multiplied by this
# ratio before they enter the model (and CODE-15% tracings are divided by it to obtain mV). The ratio is the median over the 12 leads
# of the amplitude of CODE-15% tracings divided by that of PTB-XL tracings; section 1 of notebook 04 shows its computation and the
# choice of the scale.
CODE_AMPLITUDE_RATIO = 1.7962195678461295

## The ten models

One public model trained for age (Lima et al.) and three ECG foundation models, each adapted in three ways: a frozen embedding with ridge regression
trained on MIMIC-IV-ECG ("frozen"), the whole network fine-tuned on MIMIC-IV-ECG, and the whole network fine-tuned on CODE-15%.
Every file of a model is named by its key (first column). Colours and markers are the same in every figure.

In [ ]:
import numpy as np
import pandas as pd

ADAPTATIONS = {"public": "public model", "frozen": "frozen", "mimic": "fine-tuned (MIMIC)", "code15": "fine-tuned (CODE-15%)"}
MODELS = pd.DataFrame([
    # key                   foundation model  adaptation
    ("lima",                "Lima",          "public"),
    ("ecgfounder_frozen",   "ECGFounder",    "frozen"),
    ("ecgfounder_mimic",    "ECGFounder",    "mimic"),
    ("ecgfounder_code15",   "ECGFounder",    "code15"),
    ("ecgfm_frozen",        "ECG-FM",        "frozen"),
    ("ecgfm_mimic",         "ECG-FM",        "mimic"),
    ("ecgfm_code15",        "ECG-FM",        "code15"),
    ("hubert_frozen",       "HuBERT-ECG",    "frozen"),
    ("hubert_mimic",        "HuBERT-ECG",    "mimic"),
    ("hubert_code15",       "HuBERT-ECG",    "code15"),
], columns=["key", "fm", "adapt"]).set_index("key")
MODEL_KEYS = list(MODELS.index)                     # the order of rows in every table and figure
FM_COLOR = {"Lima": "#0b0b0b", "ECGFounder": "#2a78d6", "ECG-FM": "#eb6834", "HuBERT-ECG": "#1baf7a"}
MARKER = {"public": "D", "frozen": "s", "mimic": "o", "code15": "^"}


def model_label(key):
    """Row label used in tables and figures, e.g. 'ECG-FM, fine-tuned (MIMIC)'."""
    r = MODELS.loc[key]
    return "Lima et al. model" if r.adapt == "public" else f"{r.fm}, {ADAPTATIONS[r.adapt]}"

## Shared helpers

**Each patient counts once.** Patients contribute different numbers of ECGs and of pairs. Every mean, median and error in the paper weights an ECG by
1 / (number of ECGs of its patient), and a pair by 1 / (number of pairs of its patient), so that each patient contributes equally.
Confidence intervals resample patients (patient bootstrap).

**Aggregate output only.** Credentialed data are never printed row by row; `masked()` hides counts between 1 and 9.

In [ ]:
def patient_weights(subject_id):
    """Weight 1 / (number of rows of the same patient) for every row."""
    s = pd.Series(np.asarray(subject_id))
    return (1.0 / s.map(s.value_counts())).to_numpy()


def weighted_median(x, w):
    o = np.argsort(x)
    c = np.cumsum(np.asarray(w)[o])
    return float(np.asarray(x)[o][np.searchsorted(c, c[-1] / 2)])


def masked(n):
    """A count for printing: values between 1 and SMALL_CELL - 1 are shown as '<10'."""
    n = int(n)
    return f"<{SMALL_CELL}" if 0 < n < SMALL_CELL else f"{n:,}"


def rng_for(name):
    """An independent random generator for one analysis, derived from BOOT_SEED and the name of the analysis."""
    return np.random.default_rng([BOOT_SEED, sum(ord(c) * 31 ** i for i, c in enumerate(name)) % (2 ** 32)])


def bootstrap_patients(subject_id, n_boot, rng):
    """Yield, for each resample, the row indices of a patient bootstrap (patients drawn with replacement, all their rows kept)."""
    codes, uniq = pd.factorize(pd.Series(np.asarray(subject_id)))
    order = np.argsort(codes, kind="stable")
    starts = np.r_[0, np.cumsum(np.bincount(codes))]
    n = len(uniq)
    for _ in range(n_boot):
        pick = rng.integers(0, n, n)
        yield np.concatenate([order[starts[p]:starts[p + 1]] for p in pick])


def percentile_ci(values):
    """95% percentile interval of bootstrap values (NaN-safe)."""
    v = np.asarray(values, float)
    v = v[np.isfinite(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) > 10 else (np.nan, np.nan)


def load_predictions(key):
    """AI-ECG age (and the sex logit) of one model for the included ECGs of the MIMIC-IV-ECG test split (notebook 04)."""
    return pd.read_parquet(PREDICTIONS / f"{key}.parquet")

## Data conventions and the cohorts of the mortality analysis

**Waveform arrays.** Notebook 01 stores every ECG of the three datasets in one format: a 10-s, 500-Hz recording as 12 rows of 5,000 samples (int16),
200 digital units per millivolt, leads in the order of MIMIC-IV-ECG (I, II, III, aVR, aVF, aVL, V1–V6). The standard order
(I, II, III, aVR, aVL, aVF, V1–V6) differs only by the positions of aVL and aVF, so the same index converts in both directions.
A missing sample of MIMIC-IV-ECG keeps its value in the raw files, −32768; the other two datasets have no missing samples.

**Cohorts of the mortality analysis** (notebook 02, test split): for each patient with known sex, the first ECG that is followed by another ECG
within 48 hours (`MORTALITY_COHORT`), and the first run of five ECGs within 7 days or within 48 hours (`AVERAGING_7D`, `AVERAGING_48H`).

In [ ]:
WAVE_LEADS = ["I", "II", "III", "aVR", "aVF", "aVL", "V1", "V2", "V3", "V4", "V5", "V6"]   # row order of every *_WAVE500 array
LEAD_SWAP = [0, 1, 2, 3, 5, 4, 6, 7, 8, 9, 10, 11]   # exchanges aVF and aVL: MIMIC order <-> standard order (x[:, LEAD_SWAP])
WAVE_UNITS_PER_MV = 200                              # digital units per millivolt in every *_WAVE500 array
WAVE_MISSING = -32768                                # a missing sample in MIMIC_WAVE500

CODE15_EXAMS = DERIVED / "code15_exams.parquet"      # the CODE-15% exam behind each row of the CODE-15% arrays (notebook 01; notebook 02
                                                     # adds the age rule and the patient-level split and writes CODE15_RECORDS)

MORTALITY_COHORT = SETS / "mortality_cohort.parquet"      # idx_first, idx_next, subject_id, n_ecgs (ECGs within 48 h from idx_first)
AVERAGING_7D = SETS / "averaging_within_7d.parquet"       # idx_1 ... idx_5, subject_id, n_ecgs (ECGs within 7 days from idx_1)
AVERAGING_48H = SETS / "averaging_within_48h.parquet"     # idx_1 ... idx_5, subject_id, n_ecgs (ECGs within 48 h from idx_1)

## Models: pretrained weights, network definitions and inputs

Notebooks 03 (training) and 04 (prediction) share the definitions below: where the pretrained weights and the developers' code are
placed, how each network is built, how an ECG is prepared for each model, how the large waveform arrays are read in batches, and
how accuracy is measured. Notebook 03 explains where each file comes from. `torch` and `transformers` are imported inside the
functions, so notebooks that never run a network do not need them.

| Model | Input (from the arrays of notebook 01) | Embedding |
|---|---|---|
| Lima et al. | 12 leads in mV, standard lead order, multiplied by `CODE_AMPLITUDE_RATIO`, resampled from 500 to 400 Hz (polyphase), 4,000 samples centred in 4,096 | – (the network outputs age) |
| ECGFounder | 12 leads in mV, standard lead order, 500 Hz, 10 s; one z-score over the whole 12 × 5,000 record | mean over time of the last stage (1,024) |
| ECG-FM | 12 leads in mV, standard lead order, 500 Hz; z-score of each lead over the 10 s; two 5-s halves | mean over time of the last transformer layer, averaged over the two halves (768) |
| HuBERT-ECG large | the inputs of notebook 01: two 5-s windows, band-pass 0.05–47 Hz, each lead scaled to [−1, 1], 100 Hz | mean over time of the last hidden state, averaged over the two windows (960) |

Missing samples (code −32768) are set to 0 before any scaling. Every model has two linear heads on the embedding: AI-ECG age,
predicted as (age − 60) / 10, and the logit of male sex.

In [ ]:
# --- pretrained weights, as released by their developers (notebook 03) ------------------------------------------------------------
LIMA_WEIGHTS = PRETRAINED / "lima2021" / "weights" / "model"          # config.json, model.pth (model.zip of Zenodo record 4892365)
ECGFOUNDER_WEIGHTS = PRETRAINED / "ecgfounder" / "12_lead_ECGFounder.pth"
ECGFM_WEIGHTS = PRETRAINED / "ecg-fm" / "mimic_iv_ecg_physionet_pretrained.pt"
HUBERT_WEIGHTS = PRETRAINED / "hubert-ecg" / "large"                 # config.json, model.safetensors

# --- third-party code: clones of the developers' repositories (notebook 03) ----------------------------------------------------------
LIMA_CODE = THIRD_PARTY / "ecg-age-prediction"                       # resnet.py (network of the Lima et al. model)
ECGFOUNDER_CODE = THIRD_PARTY / "ECGFounder"                         # net1d.py (network of ECGFounder)

# --- graphics card ----------------------------------------------------------------------------------------------------------------
GPU_MEMORY_FRACTION = None      # share of the GPU memory this process may use (None: no limit)

# --- embeddings and heads ---------------------------------------------------------------------------------------------------------
EMBED_DIM = {"ecgfounder": 1024, "ecgfm": 768, "hubert": 960}
AGE_CENTRE, AGE_SCALE = 60.0, 10.0   # the age head predicts (age - 60) / 10, i.e. age in decades around 60 years

# --- batch settings of the fine-tuned models (training, validation and prediction use the same batches; notebook 03) -----------
# bs: ECGs per batch; accum: batches per optimisation step; lr_bb / lr_head: learning rates of the network and the heads;
# wd: weight decay; block: rows read from disk at once (only affects speed and the shuffling, see block_batches)
FINETUNE = {
    "ecgfounder": dict(bs=128, accum=1, lr_bb=1e-4, lr_head=1e-3, wd=1e-4, block=2048),
    "ecgfm":      dict(bs=32, accum=4, lr_bb=5e-5, lr_head=1e-3, wd=1e-2, block=2048),
    "hubert":     dict(bs=32, accum=4, lr_bb=3e-5, lr_head=1e-3, wd=1e-2, block=4096),
}

# keys of the developers' HuBERT-ECG config.json that are not settings of the standard HuBERT network of transformers
HUBERT_EXTRA_KEYS = ("model_type", "architectures", "auto_map", "ensemble_length", "vocab_sizes", "id2label", "label2id",
                     "transformers_version", "torch_dtype", "dtype", "task_type", "use_label_embedding", "classifier_hidden_size",
                     "classifier_dropout_prob", "activation")


def model_parts(key):
    """Foundation model and adaptation of a model key: 'ecgfm_code15' -> ('ecgfm', 'code15'), 'lima' -> ('lima', 'public')."""
    fm, _, adapt = key.rpartition("_")
    return (fm, adapt) if fm else (key, "public")


def gpu_device():
    """The graphics card, with the memory limit GPU_MEMORY_FRACTION applied before the first allocation."""
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("this step needs a CUDA graphics card")
    if GPU_MEMORY_FRACTION:
        torch.cuda.set_per_process_memory_fraction(float(GPU_MEMORY_FRACTION), 0)
    return torch.device("cuda", 0)


def gpu_precision(tf32_matmul, tf32_conv=True, benchmark=False):
    """Numerical settings of the graphics card (TensorFloat-32 for matrix products and convolutions, cuDNN autotuning).
    Notebooks 03 and 04 set them at the start of each step that runs a network."""
    import torch
    torch.backends.cuda.matmul.allow_tf32 = tf32_matmul
    torch.backends.cudnn.allow_tf32 = tf32_conv
    torch.backends.cudnn.benchmark = benchmark


def load_checkpoint(path):
    """torch.load without executing pickled code (weights_only=True). The published checkpoints of the Lima et al. model and of
    ECGFounder also store a few numpy numbers (e.g. a validation loss); exactly these three numpy types are allowed."""
    import torch
    core = getattr(np, "_core", None) or np.core
    allowed = [(core.multiarray.scalar, "numpy.core.multiarray.scalar"), np.dtype, np.dtypes.Float64DType]
    with torch.serialization.safe_globals(allowed):
        return torch.load(path, map_location="cpu", weights_only=True)

### Network definitions

* **Lima et al.**: `ResNet1d` from the authors' `resnet.py`, built from their `config.json`.
* **ECGFounder**: `Net1D` from the developers' `net1d.py`, with the settings of their fine-tuning code (`finetune_model.py`).
* **ECG-FM**: the encoder of a wav2vec 2.0 model as defined in fairseq-signals (convolutional feature encoder of four layers with
  256 channels, kernel 2 and stride 2; layer norm; projection to 768; convolutional positional embedding of kernel 128 in 16 groups;
  12 post-norm transformer layers with 12 heads and 3,072 hidden units), written out below in plain PyTorch with the parameter names
  of the released checkpoint, which therefore loads with `strict=True`. Dropout, layer drop and masking are not used (the fine-tuning
  defaults of fairseq-signals).
* **HuBERT-ECG large**: the standard HuBERT network of `transformers`, configured by the developers' `config.json`, with the
  masking of input features (SpecAugment) switched off.

In [ ]:
_NETWORKS = {}


def _networks():
    """The torch module classes, defined on first use (so that importing this notebook does not import torch)."""
    if _NETWORKS:
        return _NETWORKS
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class GroupNorm32(nn.GroupNorm):
        """Group norm computed in float32 also under mixed precision (as fairseq's Fp32GroupNorm)."""
        def forward(self, x):
            return F.group_norm(x.float(), self.num_groups, self.weight.float(), self.bias.float(), self.eps).type_as(x)

    class FeatureEncoder(nn.Module):
        """Four convolutions (kernel 2, stride 2, no bias) with GELU; group norm after the first. 500 Hz -> 31.25 Hz."""
        def __init__(self, channels=256):
            super().__init__()
            blocks = []
            for i in range(4):
                conv = nn.Conv1d(12 if i == 0 else channels, channels, kernel_size=2, stride=2, bias=False)
                # position 1 is a dropout of probability 0 in fairseq-signals; an identity keeps the parameter names
                blocks.append(nn.Sequential(conv, nn.Identity(), GroupNorm32(channels, channels), nn.GELU()) if i == 0
                              else nn.Sequential(conv, nn.Identity(), nn.GELU()))
            self.conv_layers = nn.ModuleList(blocks)

        def forward(self, x):
            for block in self.conv_layers:
                x = block(x)
            return x

    class DropLast(nn.Module):
        """Drops the last time step: a convolution with an even kernel and padding k/2 returns one step too many."""
        def forward(self, x):
            return x[:, :, :-1]

    class PositionalConv(nn.Module):
        def __init__(self, dim=768, kernel=128, groups=16):
            super().__init__()
            self.pos_conv = nn.Sequential(nn.Conv1d(dim, dim, kernel, padding=kernel // 2, groups=groups), DropLast(), nn.GELU())

        def forward(self, x):                                     # (B, T, C) -> (B, T, C)
            return self.pos_conv(x.transpose(1, 2)).transpose(1, 2)

    class SelfAttention(nn.Module):
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.q_proj, self.k_proj, self.v_proj, self.out_proj = (nn.Linear(dim, dim) for _ in range(4))

        def forward(self, x):                                     # (B, T, C); scaled dot-product attention, scale 1/sqrt(C/heads)
            b, t, c = x.shape
            q, k, v = (p(x).view(b, t, self.heads, c // self.heads).transpose(1, 2) for p in (self.q_proj, self.k_proj, self.v_proj))
            return self.out_proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(b, t, c))

    class EncoderLayer(nn.Module):
        """Post-norm transformer layer: x = LN(x + attention(x)); x = LN(x + fc2(gelu(fc1(x)))), GELU in float32."""
        def __init__(self, dim, ffn, heads):
            super().__init__()
            self.self_attn = SelfAttention(dim, heads)
            self.self_attn_layer_norm = nn.LayerNorm(dim)
            self.fc1, self.fc2 = nn.Linear(dim, ffn), nn.Linear(ffn, dim)
            self.final_layer_norm = nn.LayerNorm(dim)

        def forward(self, x):
            x = self.self_attn_layer_norm(x + self.self_attn(x))
            return self.final_layer_norm(x + self.fc2(F.gelu(self.fc1(x).float()).type_as(x)))

    class Encoder(nn.Module):
        def __init__(self, dim, ffn, heads, n_layers):
            super().__init__()
            self.layers = nn.ModuleList(EncoderLayer(dim, ffn, heads) for _ in range(n_layers))
            self.layer_norm = nn.LayerNorm(dim)

        def forward(self, x):
            x = self.layer_norm(x)                                # post-norm encoders of fairseq normalise once before the layers
            for layer in self.layers:
                x = layer(x)
            return x

    class ECGFMEncoder(nn.Module):
        """ECG-FM: (B, 12, T) at 500 Hz -> (B, T / 16, 768), the output of the last transformer layer."""
        def __init__(self, dim=768, ffn=3072, heads=12, n_layers=12, channels=256):
            super().__init__()
            self.feature_extractor = FeatureEncoder(channels)
            self.layer_norm = nn.LayerNorm(channels)
            self.post_extract_proj = nn.Linear(channels, dim)
            self.conv_pos = PositionalConv(dim)
            self.encoder = Encoder(dim, ffn, heads, n_layers)

        def forward(self, x):
            f = self.post_extract_proj(self.layer_norm(self.feature_extractor(x).transpose(1, 2)))
            return self.encoder(f + self.conv_pos(f))

    class AgeSexModel(nn.Module):
        """A foundation model with two linear heads on its embedding: (age - 60) / 10 and the logit of male sex."""
        def __init__(self, fm, pretrained=True):
            super().__init__()
            self.fm = fm
            self.backbone = {"ecgfounder": ecgfounder_backbone, "ecgfm": ecgfm_backbone, "hubert": hubert_backbone}[fm](pretrained)
            self.dim = EMBED_DIM[fm]
            self.age = nn.Linear(self.dim, 1)
            self.sex = nn.Linear(self.dim, 1)
            for head in (self.age, self.sex):                     # both heads start at zero: initial prediction 60 years, p(male) = 0.5
                nn.init.zeros_(head.weight)
                nn.init.zeros_(head.bias)

        def prep(self, x):
            """Raw batch from the arrays (on the GPU) -> network input."""
            return {"ecgfounder": prep_ecgfounder, "ecgfm": prep_ecgfm, "hubert": prep_hubert}[self.fm](x)

        def embed(self, x):
            if self.fm == "ecgfounder":
                return self.backbone(x)[1]                        # Net1D returns (class logits, embedding)
            b = x.shape[0]
            if self.fm == "ecgfm":                                # the two 5-s halves pass as separate records
                z = self.backbone(torch.cat([x[..., :2500], x[..., 2500:5000]], 0)).mean(1)
                return (z[:b] + z[b:]) / 2
            z = self.backbone(input_values=x.reshape(b * 2, -1)).last_hidden_state.mean(1)
            return z.view(b, 2, -1).mean(1)

        def forward(self, x):
            f = self.embed(x)
            return self.age(f).squeeze(-1), self.sex(f).squeeze(-1), f

        def head_params(self):
            return list(self.age.parameters()) + list(self.sex.parameters())

    _NETWORKS.update(ECGFMEncoder=ECGFMEncoder, AgeSexModel=AgeSexModel)
    return _NETWORKS


def lima_model():
    """The Lima et al. network (authors' resnet.py) with the published weights, in evaluation mode, on the CPU."""
    import json
    import sys
    if str(LIMA_CODE) not in sys.path:
        sys.path.insert(0, str(LIMA_CODE))
    from resnet import ResNet1d
    cfg = json.loads((LIMA_WEIGHTS / "config.json").read_text())
    net = ResNet1d(input_dim=(12, cfg["seq_length"]), blocks_dim=list(zip(cfg["net_filter_size"], cfg["net_seq_lengh"])),
                   n_classes=1, kernel_size=cfg["kernel_size"], dropout_rate=cfg["dropout_rate"])
    net.load_state_dict(load_checkpoint(LIMA_WEIGHTS / "model.pth")["model"], strict=True)
    return net.eval()


def ecgfounder_backbone(pretrained=True):
    """Net1D with the settings of the developers' fine-tuning code; 150 diagnostic outputs and the 1,024-dimensional embedding."""
    import sys
    if str(ECGFOUNDER_CODE) not in sys.path:
        sys.path.insert(0, str(ECGFOUNDER_CODE))
    from net1d import Net1D
    net = Net1D(in_channels=12, base_filters=64, ratio=1, filter_list=[64, 160, 160, 400, 400, 1024, 1024],
                m_blocks_list=[2, 2, 2, 3, 3, 4, 4], kernel_size=16, stride=2, groups_width=16, verbose=False,
                use_bn=False, use_do=False, n_classes=150, return_features=True)
    if pretrained:
        net.load_state_dict(load_checkpoint(ECGFOUNDER_WEIGHTS)["state_dict"], strict=True)
    return net


def ecgfm_backbone(pretrained=True):
    """The ECG-FM encoder with the released pretrained weights. The checkpoint stores the positional convolution with weight
    normalisation (direction v and magnitude g); its weight is g * v / ||v||, the norm taken over all axes but the kernel axis,
    computed with torch._weight_norm, the function that PyTorch's own weight normalisation uses. Parts used only for pretraining
    (mask embedding, quantizer, projections of the contrastive task) are not needed."""
    import torch
    net = _networks()["ECGFMEncoder"]()
    if pretrained:
        sd = dict(load_checkpoint(ECGFM_WEIGHTS)["model"])
        g, v = sd.pop("conv_pos.pos_conv.0.weight_g"), sd.pop("conv_pos.pos_conv.0.weight_v")
        sd["conv_pos.pos_conv.0.weight"] = torch._weight_norm(v, g, 2)
        for k in [k for k in sd if k.split(".")[0] in ("mask_emb", "quantizer", "project_q", "final_proj")]:
            del sd[k]
        net.load_state_dict(sd, strict=True)
    return net


def hubert_backbone(pretrained=True):
    """HuBERT-ECG large as a standard transformers HubertModel. The released weights also hold the heads of the pretraining task
    (final_proj, label_embedding), which are not needed; masked_spec_embed (used only for masking) is not in the release."""
    import json
    from transformers import HubertConfig, HubertModel
    cfg = json.loads((HUBERT_WEIGHTS / "config.json").read_text())
    for k in HUBERT_EXTRA_KEYS:
        cfg.pop(k, None)
    config = HubertConfig(**cfg)
    config.apply_spec_augment = False                           # no masking of input features during fine-tuning
    net = HubertModel(config)
    if pretrained:
        from safetensors.torch import load_file
        sd = {k: v for k, v in load_file(str(HUBERT_WEIGHTS / "model.safetensors")).items()
              if k.split(".")[0] not in ("final_proj", "label_embedding")}
        missing, unexpected = net.load_state_dict(sd, strict=False)
        missing = [m for m in missing if m != "masked_spec_embed"]
        if missing or unexpected:
            raise RuntimeError(f"HuBERT-ECG weights do not match the network: {len(missing)} missing, {len(unexpected)} unexpected")
    return net


def age_sex_model(fm, pretrained=True):
    """Foundation model `fm` ('ecgfounder', 'ecgfm' or 'hubert') with age and sex heads (heads at zero)."""
    return _networks()["AgeSexModel"](fm, pretrained)


def load_trained(key):
    """A fine-tuned model from TRAINED/{key}/best.pt, loaded with strict=True, and the content of the checkpoint."""
    ck = load_checkpoint(TRAINED / key / "best.pt")
    model = age_sex_model(model_parts(key)[0], pretrained=False)
    model.load_state_dict(ck["model"], strict=True)
    return model, ck

### Input preparation, batches and prediction

In [ ]:
def prep_ecgfounder(x):
    """(B, 12, 5000) int16 -> mV in standard lead order, z-scored with one mean and SD over the whole record (ECGFounder)."""
    import torch
    xf = x.float()
    xf = torch.where(x == WAVE_MISSING, torch.zeros_like(xf), xf / float(WAVE_UNITS_PER_MV))[:, LEAD_SWAP]
    return (xf - xf.mean(dim=(1, 2), keepdim=True)) / (xf.std(dim=(1, 2), keepdim=True) + 1e-8)


def prep_ecgfm(x):
    """(B, 12, 5000) int16 -> mV in standard lead order, each lead z-scored over the 10 s (a constant lead becomes 0) (ECG-FM)."""
    import torch
    xf = torch.where(x == WAVE_MISSING, torch.zeros_like(x, dtype=torch.float32), x.float() / float(WAVE_UNITS_PER_MV))[:, LEAD_SWAP]
    xf = xf - xf.mean(-1, keepdim=True)
    sd = xf.std(-1, keepdim=True, unbiased=False)
    return torch.where(sd == 0, torch.zeros_like(xf), xf / sd.clamp_min(1e-12))


def prep_hubert(x):
    """(B, 2, 6000) float16 HuBERT-ECG inputs of notebook 01 -> float32."""
    return x.float()


def lima_input(x_int16, scale):
    """(B, 12, 5000) int16 -> (B, 12, 4096) float32 input of the Lima et al. model: mV in standard lead order times `scale`,
    500 -> 400 Hz by polyphase resampling, the 4,000 samples centred in 4,096 (48 zeros on each side)."""
    from scipy.signal import resample_poly
    x = np.where(x_int16 == WAVE_MISSING, 0, x_int16).astype(np.float32)[:, LEAD_SWAP] / float(WAVE_UNITS_PER_MV) * scale
    return np.pad(resample_poly(x, 4, 5, axis=-1), ((0, 0), (0, 0), (48, 48))).astype(np.float32)


def is_flat(raw):
    """True for a record that is constant over all leads and samples (CODE-15% contains some all-zero tracings)."""
    v = raw.flatten(1)
    return v.amax(1) == v.amin(1)


def model_inputs(source, fm):
    """Memory-mapped input array of a dataset ('mimic', 'code15', 'ptbxl') for a model: the HuBERT-ECG inputs for HuBERT-ECG,
    the 500 Hz int16 waveforms for all other models (row = idx for MIMIC-IV-ECG, row of the dataset otherwise)."""
    if fm == "hubert":
        path = {"mimic": MIMIC_HUBERT, "code15": CODE15_HUBERT, "ptbxl": PTBXL_HUBERT}[source]
    else:
        path = {"mimic": MIMIC_WAVE500, "code15": CODE15_WAVE500, "ptbxl": PTBXL_WAVE500}[source]
    return np.load(path, mmap_mode="r")


def block_batches(arr, rows, batch_size, shuffle=False, block=2048, seed=0, prefetch=3):
    """Yield (rows, data) batches of the records `rows` of a memory-mapped array.

    Reading single records at random positions is slow on a hard disk. The records are therefore grouped into blocks of `block`
    consecutive rows, each block is read at once, and batches of `batch_size` are cut from the stream of blocks (the last batch
    may be smaller). Without shuffling the records come in increasing order. With shuffling (training) the blocks are visited in
    a random order, two at a time, and the records of each pair are shuffled; the generator is seeded by `seed` (the epoch).
    A background thread reads ahead; it stops when the loop over the batches ends early (e.g. at an early stop)."""
    import queue
    import threading
    rows = np.sort(np.asarray(rows))
    groups = np.split(rows, np.flatnonzero(np.diff(rows // block)) + 1) if len(rows) else []
    rng = np.random.default_rng(seed)
    q, stop = queue.Queue(maxsize=prefetch), threading.Event()

    def read(r):
        chunk = np.asarray(arr[r[0]:r[-1] + 1])                  # one contiguous read
        return r, chunk[r - r[0]]

    def put(item):
        while not stop.is_set():
            try:
                q.put(item, timeout=0.5)
                return True
            except queue.Full:
                pass
        return False

    def producer():
        try:
            order = rng.permutation(len(groups)) if shuffle else np.arange(len(groups))
            step = 2 if shuffle else 1
            for i in range(0, len(order), step):
                parts = [read(groups[j]) for j in order[i:i + step]]
                r, d = np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])
                if shuffle:
                    perm = rng.permutation(len(r))
                    r, d = r[perm], d[perm]
                if not put((r, d)):
                    return
            put(None)
        except BaseException as e:                                 # hand a read error to the consumer instead of ending early
            put(e)

    threading.Thread(target=producer, daemon=True).start()
    try:
        carry_r = carry_d = None
        while True:
            item = q.get()
            if item is None:
                break
            if isinstance(item, BaseException):
                raise item
            r, d = item
            if carry_r is not None:
                r, d = np.concatenate([carry_r, r]), np.concatenate([carry_d, d])
            n_full = len(r) // batch_size * batch_size
            for s in range(0, n_full, batch_size):
                yield r[s:s + batch_size], d[s:s + batch_size]
            carry_r, carry_d = (r[n_full:], d[n_full:]) if n_full < len(r) else (None, None)
        if carry_r is not None and len(carry_r):
            yield carry_r, carry_d
    finally:
        stop.set()                                                 # lets the reader thread end and free its blocks


def predict_records(model, arr, rows, batch_size, block, device):
    """AI-ECG age (years), sex logit and the flat flag of the records `rows` (sorted) with a model with heads, in bfloat16
    mixed precision. Returns rows, age, logit, flat as numpy arrays in increasing row order."""
    import torch
    was_training = model.training
    model.eval()
    out = {"r": [], "a": [], "s": [], "f": []}
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        for r, x in block_batches(arr, rows, batch_size, block=block):
            raw = torch.from_numpy(np.ascontiguousarray(x)).to(device, non_blocking=True)
            a, s, _ = model(model.prep(raw))
            out["r"].append(r)
            out["a"].append(a.float().cpu().numpy())
            out["s"].append(s.float().cpu().numpy())
            out["f"].append(is_flat(raw).cpu().numpy())
    model.train(was_training)
    r = np.concatenate(out["r"])
    o = np.argsort(r)
    return r[o], np.concatenate(out["a"])[o] * AGE_SCALE + AGE_CENTRE, np.concatenate(out["s"])[o], np.concatenate(out["f"])[o]


def probe_predict(emb, probe):
    """Age (years) and sex logit from frozen embeddings (rows of EMBEDDINGS) with a fitted probe (the content of ridge.npz)."""
    x = np.array(emb, dtype=np.float32)
    x -= probe["feature_mean"]
    x /= probe["feature_sd"]
    return x @ probe["age_coef"] + probe["age_intercept"], x @ probe["sex_coef"] + probe["sex_intercept"]

### Labels and accuracy

Age and sex used for training and evaluation. In MIMIC-IV-ECG they come from MIMIC-IV (credentialed, notebook 02) and only included
ECGs are used; in CODE-15% and PTB-XL from the files of the datasets (notebook 01), at ages of 18–88 years. The MAE and the
correlation weight each ECG by 1 / (number of ECGs of its patient among the ECGs evaluated), so that every patient counts equally;
this MAE also chooses the ridge penalty and the checkpoint.

In [ ]:
def model_labels(source):
    """DataFrame indexed by the row of the arrays (idx for MIMIC-IV-ECG): age (NaN where not usable), male (1/0, NaN if
    unknown), split (train/val/test; NaN where the record is not used) and pid (patient)."""
    if source == "mimic":
        coh = pd.read_parquet(COHORT, columns=["idx", "subject_id", "split", "include"])
        coh = coh[coh["include"]].set_index("idx")
        ctx = pd.read_parquet(ECG_CONTEXT, columns=["idx", "age", "sex"]).set_index("idx")
        df = pd.DataFrame({"split": coh["split"], "pid": coh["subject_id"]}).join(ctx)
        df["male"] = (df["sex"] == "M").astype(float).where(df["sex"].notna())
        return df[["age", "male", "split", "pid"]]
    lab = pd.read_parquet({"code15": CODE15_RECORDS, "ptbxl": PTBXL_RECORDS}[source]).set_index("row")
    valid = lab["age_valid"].fillna(False).astype(bool)
    out = pd.DataFrame({"age": lab["age"].where(valid), "male": lab["is_male"].astype(float), "pid": lab["patient_id"]})
    # PTB-XL is used only for evaluation: every record at ages 18-88 counts as 'test'
    out["split"] = lab["split"] if source == "code15" else pd.Series("test", index=lab.index).where(valid)
    return out


def age_metrics(age, pred, pid, male=None, logit=None):
    """Accuracy of AI-ECG age with each patient weighted equally (MAE and correlation), counts, and the AUC for sex
    (logit against male, same patient weights). NaN ages and sexes are left out."""
    from sklearn.metrics import roc_auc_score
    age, pred, pid = np.asarray(age, float), np.asarray(pred, float), np.asarray(pid)
    ok = ~np.isnan(age)
    out = {"n_age": int(ok.sum())}
    if ok.sum() >= 10:
        d = pred[ok] - age[ok]
        w = patient_weights(pid[ok])
        cv = np.cov(pred[ok], age[ok], aweights=w)
        out.update(mae=float(np.average(np.abs(d), weights=w)), r=float(cv[0, 1] / np.sqrt(cv[0, 0] * cv[1, 1])),
                   n_patients=int(len(np.unique(pid[ok]))))
    if male is not None:
        male, logit = np.asarray(male, float), np.asarray(logit, float)
        ok = ~np.isnan(male)
        out["n_sex"] = int(ok.sum())
        if ok.sum() >= 10 and 0 < male[ok].mean() < 1:
            out["sex_auc"] = float(roc_auc_score(male[ok], logit[ok], sample_weight=patient_weights(pid[ok])))
    return out

## Display style of the tables and figures

Notebooks 07 and 08 draw the tables and figures of the paper with the conventions defined here.

* **Figures** are laid out in millimetres on the two-column width of 180 mm, in Arial (or the first of the fonts in `FIGURE_FONTS` that is
  installed) at 4–8 pt, with bold lower-case panel letters. Each figure is saved in `FIGURES` as a PNG at 300 dpi, a PDF and a CSV file with
  the plotted values (`<name>_source.csv`, the source data of the paper).
* **Tables** are written to `TABLES` as markdown and as CSV. Numbers are rounded half away from zero,
  negative numbers carry a true minus sign, counts from 1,000 a thousands separator, and a 95% confidence interval goes on a second line
  of its cell.
* Files that already exist are kept unless the switch `OVERWRITE` of notebook 07 or 08 is `True`, so that a finished output is never
  replaced by accident.

matplotlib is imported inside the functions only, so that the other notebooks do not load it.

In [ ]:
from decimal import ROUND_HALF_UP, Decimal

MM = 1 / 25.4                                                       # inches per millimetre
FIGURE_WIDTH = 180.0                                                # two-column width (mm)
FIGURE_FONTS = ("Arial", "Helvetica", "Liberation Sans", "Arimo", "DejaVu Sans")   # the first one installed is used
INK, INK2, GRID, AXIS = "#0b0b0b", "#52514e", "#e6e5df", "#6b6a65"   # text, secondary text, grid and separator lines, axes
GREY, BAND = "#898781", "#ecebe6"                                   # markers that stand for an adaptation; shaded reference bands


def setup_figures():
    """matplotlib settings for print (font sizes in pt, line widths in pt); returns the font family in use."""
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    installed = {f.name for f in font_manager.fontManager.ttflist}
    family = next((f for f in FIGURE_FONTS if f in installed), "DejaVu Sans")
    plt.rcParams.update({
        "font.family": family, "font.size": 6, "axes.labelsize": 6, "axes.titlesize": 6.5, "xtick.labelsize": 5.5, "ytick.labelsize": 5.5,
        "legend.fontsize": 5.5, "axes.linewidth": 0.5, "axes.edgecolor": AXIS, "axes.labelcolor": INK, "text.color": INK,
        "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK, "ytick.labelcolor": INK,
        "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2, "ytick.major.size": 2,
        "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold", "axes.titlepad": 3, "axes.labelpad": 2,
        "lines.linewidth": 0.9, "lines.markersize": 3, "errorbar.capsize": 0, "legend.frameon": False,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 150,
    })
    return family


def figure_mm(h_mm, w_mm=FIGURE_WIDTH):
    """An empty figure of w_mm x h_mm millimetres."""
    import matplotlib.pyplot as plt
    return plt.figure(figsize=(w_mm * MM, h_mm * MM))


def axes_mm(fig, x, y, w, h, **kw):
    """Axes placed in millimetres from the top-left corner of the figure: (x, y) is the top-left corner of the axes, w and h their size."""
    W, H = fig.get_size_inches() / MM
    return fig.add_axes([x / W, 1 - (y + h) / H, w / W, h / H], **kw)


def panel_letter(fig, x, y, letter):
    """Bold lower-case panel letter with its top-left corner at (x, y) mm."""
    W, H = fig.get_size_inches() / MM
    fig.text(x / W, 1 - y / H, letter, fontsize=8, fontweight="bold", va="top", ha="left")


def save_figure(fig, name, source, overwrite=False):
    """Write FIGURES/<name>.png (300 dpi), <name>.pdf and <name>_source.csv (the plotted values, a DataFrame)."""
    import matplotlib.pyplot as plt
    files = [FIGURES / f"{name}.png", FIGURES / f"{name}.pdf", FIGURES / f"{name}_source.csv"]
    if not overwrite and any(f.exists() for f in files):
        plt.close(fig)
        print(f"{name}: files exist, kept (OVERWRITE = True draws the figure again)")
        return
    fig.savefig(files[1])
    fig.savefig(files[0], dpi=300)
    plt.close(fig)
    source.to_csv(files[2], index=False)
    print(f"{name}: saved .png, .pdf and _source.csv ({len(source)} rows of plotted values)")


def fmt_num(x, nd=1):
    """x rounded half away from zero to nd decimals, as text with a true minus sign and a thousands separator; '—' when missing."""
    if x is None or not np.isfinite(float(x)):
        return "—"
    v = Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP)
    v = abs(v) if v == 0 else v                                     # no "−0.0"
    return (f"{v:,}" if abs(v) >= 1000 else f"{v}").replace("-", "−")


def fmt_cell(v, lo, hi, nd=1):
    """An estimate with its 95% confidence interval on the second line of the table cell."""
    return f"{fmt_num(v, nd)}<br>({fmt_num(lo, nd)} to {fmt_num(hi, nd)})"


def write_table(name, header, rows, overwrite=False):
    """Write TABLES/<name>.md and <name>.csv. header: column names; rows: lists of cells, or a string for a bold group label.
    The CSV holds the same cells without markdown (line breaks inside cells become spaces)."""
    files = [TABLES / f"{name}.md", TABLES / f"{name}.csv"]
    if not overwrite and any(f.exists() for f in files):
        print(f"{name}: files exist, kept (OVERWRITE = True writes the table again)")
        return
    md = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
    md += ["| **" + r + "** |" + " |" * (len(header) - 1) if isinstance(r, str) else "| " + " | ".join(r) + " |" for r in rows]
    files[0].write_text("\n".join(md) + "\n", encoding="utf-8")
    plain = [[r] + [""] * (len(header) - 1) if isinstance(r, str) else list(r) for r in rows]
    plain = [[c.replace("<br>", " ").replace("**", "").replace("*n*", "n").strip() for c in r] for r in plain]
    pd.DataFrame(plain, columns=header).to_csv(files[1], index=False, encoding="utf-8")
    print(f"{name}: saved .md and .csv ({len(rows)} rows)")